# Assignment 2 — training pipeline

Run with `uv run python main.py train`. Start to finish, with no manual steps, this notebook:

1. Downloads **Hard Hat Workers v5** from Roboflow Universe (`joseph-nelson / hard-hat-workers / 5`).
2. Describes it from the labels: instances per class, positive images per split, heads per image, object size.
3. Scores two baselines at image level: predicting nothing / the majority class, and COCO `yolo11n` zero-shot.
4. Finetunes `yolo11n` (30 epochs, 640 px) and saves the weights to `weights/hardhat_yolo11n.pt`.
5. Evaluates on test: overall, per class, box confusion matrix, and where it fails (object size, crowd, mixed scenes).
6. Turns boxes into one decision per snapshot, sweeps the threshold on validation with the cost matrix in `costs.py`, and checks it once on test.
7. Searches the two thresholds of the pass / review / alert design, within the review time available.
8. Saves everything the dashboard needs to `artifacts/`.

The decisions behind every step are in REPORT.md. I trained the committed weights on Google Colab (T4 GPU) with
exactly this code. Setting `REUSE_WEIGHTS=1` skips step 4 and re-runs everything else on the committed weights,
which is useful on a laptop without a GPU. By default it trains.

In [ ]:
import json
import os
import shutil
from pathlib import Path

import matplotlib
matplotlib.use("Agg")            # never open a window: this must run unattended
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import yaml
from PIL import Image, ImageDraw, ImageFont
from ultralytics import YOLO

import costs
from api_key import load_api_key

ROOT = Path.cwd()                # main.py runs this from the repo root
DATA_ROOT = ROOT / "datasets"    # in .gitignore
ART_DIR = ROOT / "artifacts"
WEIGHTS_DIR = ROOT / "weights"
EXAMPLES_DIR = ROOT / "examples"
RUNS_DIR = ROOT / "runs"         # Ultralytics output, in .gitignore
for d in (ART_DIR, WEIGHTS_DIR, EXAMPLES_DIR):
    d.mkdir(exist_ok=True)
WEIGHTS = WEIGHTS_DIR / "hardhat_yolo11n.pt"

SEED = 0
EPOCHS = int(os.environ.get("EPOCHS", 30))
IMGSZ = 640
BATCH = 16
REUSE_WEIGHTS = os.environ.get("REUSE_WEIGHTS") == "1"

if torch.cuda.is_available():
    DEVICE = 0
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"
print("torch", torch.__version__, "| device", DEVICE, "| reuse weights:", REUSE_WEIGHTS)

## 1. Download the dataset

Public dataset, used as published (no fork). Version 5 is `raw_HeadHelmetClasses`: original images, a 70/20/10 split,
and only the `head` and `helmet` classes (the `person` class, which the dataset says is labelled only in some images, is dropped).

In [ ]:
WORKSPACE, PROJECT, VERSION = "joseph-nelson", "hard-hat-workers", 5
DATA_DIR = DATA_ROOT / f"{PROJECT}-{VERSION}"

if not (DATA_DIR / "data.yaml").exists():
    from roboflow import Roboflow
    rf = Roboflow(api_key=load_api_key())
    rf.workspace(WORKSPACE).project(PROJECT).version(VERSION).download("yolov11", location=str(DATA_DIR))

# Roboflow writes relative paths (../train/images) that Ultralytics resolves from wherever it is launched.
# Rewrite them as absolute so training does not depend on the working directory.
cfg = yaml.safe_load((DATA_DIR / "data.yaml").read_text())
CLASS_NAMES = list(cfg["names"].values()) if isinstance(cfg["names"], dict) else list(cfg["names"])
DATA_YAML = DATA_DIR / "data_abs.yaml"
DATA_YAML.write_text(yaml.safe_dump({
    "path": str(DATA_DIR), "train": "train/images", "val": "valid/images", "test": "test/images",
    "names": {i: n for i, n in enumerate(CLASS_NAMES)},
}, sort_keys=False))

HEAD, HELMET = CLASS_NAMES.index("head"), CLASS_NAMES.index("helmet")
SPLITS = ["train", "valid", "test"]
print(CLASS_NAMES, "| positive class: head")

## 2. Describe the dataset from the labels

One YOLO label line per box: `class x_centre y_centre width height`, as fractions of the image. Besides instances per class I care
about the **image level**, because one snapshot is the unit of decision: an image is positive if it has at least one `head`.
Object size is reported as the share of the image and as the side in pixels once YOLO rescales the long side to 640
(COCO calls an object "small" under 32x32 px).

In [ ]:
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def read_labels(path):
    if not path.exists():
        return []
    rows = []
    for line in path.read_text().splitlines():
        parts = line.split()
        if len(parts) == 5:
            rows.append((int(parts[0]), *map(float, parts[1:])))
    return rows


img_rows, box_rows = [], []
for split in SPLITS:
    for img in sorted((DATA_DIR / split / "images").iterdir()):
        if img.suffix.lower() not in IMG_EXT:
            continue
        labels = read_labels(DATA_DIR / split / "labels" / f"{img.stem}.txt")
        W, H = Image.open(img).size
        scale = IMGSZ / max(W, H)
        n_head = sum(c == HEAD for c, *_ in labels)
        n_helmet = sum(c == HELMET for c, *_ in labels)
        img_rows.append({"split": split, "image": img.name, "path": str(img), "width": W, "height": H,
                         "n_boxes": len(labels), "n_head": n_head, "n_helmet": n_helmet, "positive": int(n_head > 0)})
        for c, xc, yc, w, h in labels:
            box_rows.append({"split": split, "image": img.name, "cls": CLASS_NAMES[c], "xc": xc, "yc": yc, "w": w, "h": h,
                             "area_frac": w * h, "side_px_640": np.sqrt(w * W * h * H) * scale})

images, boxes = pd.DataFrame(img_rows), pd.DataFrame(box_rows)

# sanity checks
print(images["split"].value_counts().reindex(SPLITS).to_string())
assert boxes[["xc", "yc", "w", "h"]].apply(lambda s: s.between(0, 1)).all().all(), "coordinates outside [0, 1]"
assert not images.duplicated(["split", "image"]).any()
print("images without any box:", int((images["n_boxes"] == 0).sum()))

In [ ]:
inst = pd.crosstab(boxes["cls"], boxes["split"]).reindex(columns=SPLITS)
inst["total"] = inst.sum(axis=1)
inst["% of total"] = (inst["total"] / inst["total"].sum() * 100).round(1)

img_level = images.groupby("split").agg(
    images=("image", "count"), positive_images=("positive", "sum"), positive_rate=("positive", "mean"),
    heads_per_image=("n_head", "mean"), helmets_per_image=("n_helmet", "mean"), boxes_per_image=("n_boxes", "mean"),
).reindex(SPLITS)

# what kind of positive image: nobody has a helmet (crowds) vs someone without a helmet among people who wear one
images["scene"] = np.select(
    [images["positive"] == 0, images["n_helmet"] == 0, images["n_head"] == 1],
    ["negative (all helmets)", "positive: nobody wears a helmet", "positive: 1 head among helmets"],
    default="positive: several heads among helmets",
)
scenes = pd.crosstab(images["scene"], images["split"]).reindex(columns=SPLITS)

heads_bucket = pd.crosstab(pd.cut(images["n_head"], [-1, 0, 1, 5, 15, 10_000], labels=["0", "1", "2-5", "6-15", ">15"]),
                           images["split"]).reindex(columns=SPLITS)

size = boxes.groupby("cls").agg(
    n=("area_frac", "size"),
    area_median_pct=("area_frac", lambda s: s.median() * 100),
    area_p10_pct=("area_frac", lambda s: s.quantile(0.10) * 100),
    area_p90_pct=("area_frac", lambda s: s.quantile(0.90) * 100),
    side_px_median=("side_px_640", "median"),
    small_lt_32px=("side_px_640", lambda s: (s < 32).mean()),
    tiny_lt_16px=("side_px_640", lambda s: (s < 16).mean()),
)

print(inst.to_string(), "\n")
print(img_level.round(3).to_string(), "\n")
print(scenes.to_string(), "\n")
print(heads_bucket.to_string(), "\n")
print(size.round(3).to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for c in CLASS_NAMES:
    axes[0].hist(boxes.loc[boxes["cls"] == c, "side_px_640"], bins=60, range=(0, 300), histtype="step", lw=2, label=c)
axes[0].axvline(32, color="k", ls="--", lw=1, label="32 px (COCO 'small')")
axes[0].set(xlabel="box side at 640 px", ylabel="boxes", title="Object size")
axes[0].legend()
scenes.drop(index="negative (all helmets)").T.plot.bar(stacked=True, ax=axes[1], rot=0)
axes[1].set(title="Positive images by type of scene", ylabel="images")
plt.tight_layout()
plt.savefig(ART_DIR / "dataset_size_and_scenes.png", dpi=120)
plt.close()

## 3. From boxes to one decision per snapshot, and the baselines

**Score of a snapshot** = confidence of its most confident `head` box, 0 if there is none. Alert when score ≥ threshold.
The maximum and not the mean: one person without a helmet is enough, and averaging with the helmets in the same image
would dilute exactly the evidence that matters. I predict with `conf=0.01` and set the threshold myself afterwards.

Baselines on the same splits:
- **Predict nothing** (every snapshot passes). Since most training images are negative, the majority class is the same thing.
- **COCO `yolo11n` zero-shot.** COCO has no `head` or `helmet` class, so at box level its AP is 0 by construction (the class list
  is part of the architecture). Its closest class is `person`, so at image level the baseline is "alert if there is a person",
  scored by the most confident person box.

In [ ]:
def score_images(model, paths, class_ids, conf=0.01, chunk=32):
    # returns (scores, boxes per image); score = max confidence among class_ids
    scores, all_boxes = [], []
    for i in range(0, len(paths), chunk):
        for r in model.predict(paths[i:i + chunk], conf=conf, imgsz=IMGSZ, device=DEVICE, verbose=False):
            cls = r.boxes.cls.cpu().numpy().astype(int)
            cf = r.boxes.conf.cpu().numpy()
            mask = np.isin(cls, class_ids)
            scores.append(float(cf[mask].max()) if mask.any() else 0.0)
            all_boxes.append({"cls": cls, "conf": cf, "xywhn": r.boxes.xywhn.cpu().numpy()})
    return np.array(scores), all_boxes


val_df = images[images["split"] == "valid"].reset_index(drop=True)
test_df = images[images["split"] == "test"].reset_index(drop=True)

coco = YOLO("yolo11n.pt")
PERSON = [k for k, v in coco.names.items() if v == "person"]
val_df["coco_score"], _ = score_images(coco, val_df["path"].tolist(), PERSON)
test_df["coco_score"], _ = score_images(coco, test_df["path"].tolist(), PERSON)

majority = int(images.loc[images["split"] == "train", "positive"].mean() >= 0.5)
y_test = test_df["positive"].to_numpy()
baselines = pd.DataFrame([
    {"model": "predict nothing (all pass)", **costs.confusion(y_test, np.zeros(len(y_test)), 0.5), "auc": np.nan},
    {"model": f"majority class of train ({'positive' if majority else 'negative'})",
     **costs.confusion(y_test, np.full(len(y_test), float(majority)), 0.5), "auc": np.nan},
    {"model": "COCO zero-shot: alert if a person (conf 0.25)",
     **costs.confusion(y_test, test_df["coco_score"], 0.25), "auc": costs.auc(y_test, test_df["coco_score"])},
])
print(baselines.round(3).to_string(index=False))

## 4. Finetune `yolo11n`

From the COCO weights, not from scratch: the backbone already turns pixels into edges, textures and head shapes, and what
changes is mostly the detection head (80 classes → 2). `yolo11n` because it trains in about 40 minutes on a T4 and runs on CPU in
the dashboard; 640 px because heads are small (a quarter are under 32 px); 30 epochs because there are small objects and more images
than in the class example (15). Ultralytics keeps `best.pt` by validation fitness, so validation is already "used" and the final
numbers are on test.

In [ ]:
TRAIN_SETTINGS = {"model": "yolo11n.pt", "epochs": EPOCHS, "imgsz": IMGSZ, "batch": BATCH, "seed": SEED,
                  "where": "Google Colab, T4 GPU (committed weights)"}

if REUSE_WEIGHTS and WEIGHTS.exists():
    print("REUSE_WEIGHTS=1: skipping training, using", WEIGHTS)
    RUN_DIR = None
else:
    trainer_model = YOLO("yolo11n.pt")
    trainer_model.train(data=str(DATA_YAML), epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, device=DEVICE,
                        project=str(RUNS_DIR), name="hardhat", exist_ok=True, seed=SEED, deterministic=False)
    RUN_DIR = Path(trainer_model.trainer.save_dir)
    shutil.copy(RUN_DIR / "weights" / "best.pt", WEIGHTS)
    shutil.copy(RUN_DIR / "results.png", ART_DIR / "results.png")
    TRAIN_SETTINGS["where"] = f"this machine ({DEVICE})"

model = YOLO(WEIGHTS)              # from the file, exactly as the dashboard loads it
print(WEIGHTS, f"{WEIGHTS.stat().st_size / 1e6:.1f} MB", model.names)

## 5. Test: overall, per class, and the confusion matrix

`val` with its default `conf=0.001` gives AP over every threshold. A confusion matrix at that cut is full of junk boxes, so the
matrix is computed separately at `conf=0.25`. Ultralytics' matrix is **rows = predicted, columns = true**, with `background` last.

In [ ]:
test_metrics = model.val(data=str(DATA_YAML), split="test", imgsz=IMGSZ, batch=BATCH, device=DEVICE,
                         project=str(RUNS_DIR), name="hardhat_test", exist_ok=True, plots=True)
TEST_DIR = Path(test_metrics.save_dir)
overall = {k.replace("metrics/", "").replace("(B)", ""): round(float(v), 4)
           for k, v in test_metrics.results_dict.items() if k.startswith("metrics/")}

n_inst = boxes[boxes["split"] == "test"]["cls"].value_counts()
n_img = boxes[boxes["split"] == "test"].groupby("cls")["image"].nunique()
per_class = pd.DataFrame([
    {"class": CLASS_NAMES[c], "images": int(n_img.get(CLASS_NAMES[c], 0)), "instances": int(n_inst.get(CLASS_NAMES[c], 0)),
     "precision": float(test_metrics.box.p[i]), "recall": float(test_metrics.box.r[i]),
     "AP50": float(test_metrics.box.ap50[i]), "AP50-95": float(test_metrics.box.ap[i])}
    for i, c in enumerate(test_metrics.ap_class_index)
]).set_index("class")
per_class.loc["all (mean)"] = [per_class["images"].sum(), per_class["instances"].sum(), overall["precision"],
                               overall["recall"], overall["mAP50"], overall["mAP50-95"]]

cm_metrics = model.val(data=str(DATA_YAML), split="test", imgsz=IMGSZ, batch=BATCH, device=DEVICE, conf=0.25,
                       project=str(RUNS_DIR), name="hardhat_test_cm", exist_ok=True, plots=True, verbose=False)
CM_DIR = Path(cm_metrics.save_dir)
cm_labels = CLASS_NAMES + ["background"]
cm_df = pd.DataFrame(cm_metrics.confusion_matrix.matrix.astype(int),
                     index=[f"pred {n}" for n in cm_labels], columns=[f"true {n}" for n in cm_labels])

head = per_class.loc["head"]
det_table = pd.DataFrame([
    {"model": "predict nothing", "mAP50": 0.0, "mAP50-95": 0.0, "head AP50": 0.0, "head recall": 0.0},
    {"model": "COCO yolo11n zero-shot (no head/helmet class)", "mAP50": 0.0, "mAP50-95": 0.0, "head AP50": 0.0, "head recall": 0.0},
    {"model": "yolo11n finetuned", "mAP50": overall["mAP50"], "mAP50-95": overall["mAP50-95"],
     "head AP50": head["AP50"], "head recall": head["recall"]},
]).set_index("model")

print(per_class.round(3).to_string(), "\n")
print(cm_df.to_string(), "\n")
print(det_table.round(3).to_string())

## 6. Where it fails

Image-level scores of the finetuned model on validation and test, then three cuts of the errors:
- **head recall by size**: each true `head` in test matched to a predicted `head` (conf ≥ 0.25, IoU ≥ 0.5, greedy by confidence,
  each true box claimed once);
- **head recall by crowd** (heads in the same image);
- **image recall by type of scene**, at the operating threshold chosen in section 7. This is the one that matters for a site,
  where the typical violation is one worker without a helmet among colleagues who wear one.

In [ ]:
val_df["score"], _ = score_images(model, val_df["path"].tolist(), [HEAD])
test_df["score"], test_pred_boxes = score_images(model, test_df["path"].tolist(), [HEAD])
for name, df in [("valid", val_df), ("test", test_df)]:
    print(f"{name}: image-level AUC finetuned {costs.auc(df['positive'], df['score']):.3f} | "
          f"COCO person {costs.auc(df['positive'], df['coco_score']):.3f}")


def iou_xywh(a, b):
    ax0, ay0, ax1, ay1 = a[0] - a[2] / 2, a[1] - a[3] / 2, a[0] + a[2] / 2, a[1] + a[3] / 2
    bx0, by0, bx1, by1 = b[0] - b[2] / 2, b[1] - b[3] / 2, b[0] + b[2] / 2, b[1] + b[3] / 2
    inter = max(0, min(ax1, bx1) - max(ax0, bx0)) * max(0, min(ay1, by1) - max(ay0, by0))
    union = a[2] * a[3] + b[2] * b[3] - inter
    return inter / union if union > 0 else 0.0


DET_CONF = 0.25
gt_rows = []
test_heads = boxes[(boxes["split"] == "test") & (boxes["cls"] == "head")]
for i, row in test_df.iterrows():
    gts = test_heads[test_heads["image"] == row["image"]]
    pb = test_pred_boxes[i]
    keep = (pb["cls"] == HEAD) & (pb["conf"] >= DET_CONF)
    preds = sorted(zip(pb["conf"][keep], pb["xywhn"][keep]), key=lambda t: -t[0])
    gt = gts[["xc", "yc", "w", "h"]].to_numpy()
    found = np.zeros(len(gt), dtype=bool)
    for _, p in preds:
        ious = [0.0 if found[k] else iou_xywh(p, g) for k, g in enumerate(gt)]
        if ious and max(ious) >= 0.5:
            found[int(np.argmax(ious))] = True
    for k, side in enumerate(gts["side_px_640"]):
        gt_rows.append({"image": row["image"], "side_px_640": side, "heads_in_image": row["n_head"],
                        "helmets_in_image": row["n_helmet"], "found": bool(found[k])})

gt_heads = pd.DataFrame(gt_rows)
gt_heads["size"] = pd.cut(gt_heads["side_px_640"], [0, 16, 32, 64, 10_000], labels=["<16 px", "16-32 px", "32-64 px", ">64 px"])
gt_heads["crowd"] = pd.cut(gt_heads["heads_in_image"], [0, 1, 5, 15, 10_000], labels=["1 head", "2-5", "6-15", ">15"])
recall_by_size = gt_heads.groupby("size", observed=False)["found"].agg(["mean", "count"])
recall_by_crowd = gt_heads.groupby("crowd", observed=False)["found"].agg(["mean", "count"])
print(f"\nhead recall (conf {DET_CONF}, IoU 0.5): {gt_heads['found'].mean():.3f} over {len(gt_heads)} heads")
print(recall_by_size.round(3).to_string(), "\n")
print(recall_by_crowd.round(3).to_string())

## 7. Cost of each threshold, and the operating point

The scenario, costs and the person in the loop are in `costs.py` (and justified in REPORT.md): 1,200 snapshots a day, base rate 5%,
TP = FP = 3 €, FN = 25 €, a review costs 0.15 € with 90% recall and 98% specificity, and 1 hour of review a day is available.

Zone rates are measured separately on positive and negative snapshots (properties of the model) and re-weighted with the real
base rate (5%), not the dataset's (~19%). Thresholds are **chosen on validation and checked once on test**, as in assignment 1.

In [ ]:
CUTS = np.round(np.arange(0.0, 1.0001, 0.01), 2)
NEVER = costs.NEVER


def sweep(df, **kw):
    pos, neg = costs.split_scores(df)
    rows = []
    for cut in CUTS:
        s = costs.confusion(df["positive"], df["score"], cut)
        rows.append({**{k: s[k] for k in ("cut", "recall", "specificity", "precision", "accuracy")},
                     **costs.design_cost(pos, neg, cut, cut, **kw)})
    return pd.DataFrame(rows)


def f1(s):
    return 2 * s["precision"] * s["recall"] / (s["precision"] + s["recall"])


sweep_val, sweep_test = sweep(val_df), sweep(test_df)
best = sweep_val.loc[sweep_val["eur_per_1000"].idxmin()]
CUT = float(best["cut"])
F1_CUT = float(sweep_val.loc[f1(sweep_val).idxmax(), "cut"])
ACC_CUT = float(sweep_val.loc[sweep_val["accuracy"].idxmax(), "cut"])
at_cut_test = sweep_test.loc[sweep_test["cut"] == CUT].iloc[0]
best_test = sweep_test.loc[sweep_test["eur_per_1000"].idxmin()]

pos_v, neg_v = costs.split_scores(val_df)
reference = {
    "do nothing": costs.design_cost(pos_v, neg_v, NEVER, NEVER)["eur_per_1000"],
    "person checks everything": costs.design_cost(pos_v, neg_v, 0.0, NEVER)["eur_per_1000"],
    "floor (perfect model)": costs.BASE_RATE * costs.C_TP * costs.PER,
}
print(f"cheapest threshold on validation: {CUT:.2f} -> {best['eur_per_1000']:.2f} EUR / 1,000 snapshots")
print(f"F1 would pick {F1_CUT:.2f}, accuracy would pick {ACC_CUT:.2f}")
print(f"test at {CUT:.2f}: {at_cut_test['eur_per_1000']:.2f} EUR / 1,000 (recall {at_cut_test['recall']:.3f}, "
      f"specificity {at_cut_test['specificity']:.3f}); best possible on test {best_test['cut']:.2f} -> {best_test['eur_per_1000']:.2f}")
print({k: round(v, 2) for k, v in reference.items()})

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(sweep_val["cut"], sweep_val["eur_per_1000"], lw=2, label="validation (chosen here)")
ax.plot(sweep_test["cut"], sweep_test["eur_per_1000"], lw=1.5, ls="--", label="test (checked once)")
for (name, value), color in zip(reference.items(), ["grey", "purple", "green"]):
    ax.axhline(value, color=color, ls=":", label=name)
ax.scatter([CUT], [best["eur_per_1000"]], color="red", zorder=5, label=f"chosen: {CUT:.2f}")
ax.set(xlabel="threshold on the snapshot score", ylabel="EUR per 1,000 snapshots", ylim=(0, None),
       title=f"Cost against threshold (base rate {costs.BASE_RATE:.0%}, FN {costs.C_FN:.0f} EUR, FP {costs.C_FP:.0f} EUR)")
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(ART_DIR / "cost_curve.png", dpi=120)
plt.close()

### Which assumption changes the answer

The base rate moves precision (not recall or specificity); the FN cost is the number I am least sure about; and the type of scene
says whether the dataset's positives look like a site's.

In [ ]:
rec, spec = best["recall"], best["specificity"]
base_rate_table = pd.DataFrame([
    {"base rate": p, "alert precision": p * rec / (p * rec + (1 - p) * (1 - spec)),
     "false alerts per true one": (1 - p) * (1 - spec) / (p * rec), "accuracy": p * rec + (1 - p) * spec,
     "accuracy of 'never alert'": 1 - p}
    for p in [float(val_df["positive"].mean()), 0.20, 0.10, 0.05, 0.02, 0.01]
])


def best_single(df, **kw):
    return sweep(df, **kw).sort_values("eur_per_1000").iloc[0]


rows = []
for c_fn in [3, 5, 10, 15, 25, 50, 100, 250]:
    r = best_single(val_df, c_fn=c_fn)
    rows.append({"FN cost": c_fn, "best cut": r["cut"], "recall": r["recall"], "specificity": r["specificity"],
                 "model alone": r["eur_per_1000"],
                 "do nothing": costs.design_cost(pos_v, neg_v, NEVER, NEVER, c_fn=c_fn)["eur_per_1000"],
                 "person checks all": costs.design_cost(pos_v, neg_v, 0.0, NEVER, c_fn=c_fn)["eur_per_1000"]})
sens_fn = pd.DataFrame(rows)

rows = []
for p in [0.01, 0.02, 0.05, 0.10, 0.20]:
    r = best_single(val_df, p=p)
    rows.append({"base rate": p, "best cut": r["cut"], "recall": r["recall"], "specificity": r["specificity"],
                 "model alone": r["eur_per_1000"],
                 "do nothing": costs.design_cost(pos_v, neg_v, NEVER, NEVER, p=p)["eur_per_1000"],
                 "person checks all": costs.design_cost(pos_v, neg_v, 0.0, NEVER, p=p)["eur_per_1000"]})
sens_p = pd.DataFrame(rows)

print(base_rate_table.round(3).to_string(index=False), "\n")
print(sens_fn.round(2).to_string(index=False), "\n")
print(sens_p.round(2).to_string(index=False))

## 8. Pass, review or alert

Grid of 0.02 for `low` and `high` on validation. A person is a second classifier with a price, so review only pays where they beat
the model. Two answers: the cheapest design with unlimited review, and the cheapest one that fits in the hour of review available,
which is the one I keep.

In [ ]:
GRID = np.round(np.arange(0.0, 1.0001, 0.02), 2)


def all_bands(df, **kw):
    pos, neg = costs.split_scores(df)
    return pd.DataFrame([costs.design_cost(pos, neg, lo, hi, **kw) for i, lo in enumerate(GRID) for hi in GRID[i:]])


def best_band(df, **kw):
    ds = all_bands(df, **kw)
    ok = ds[ds["review_hours_per_day"] <= costs.REVIEW_HOURS_AVAILABLE]
    return ds.loc[ds["eur_per_1000"].idxmin()], ok.loc[ok["eur_per_1000"].idxmin()]


cheapest, chosen = best_band(val_df)
LOW, HIGH = float(chosen["low"]), float(chosen["high"])


def four_designs(df):
    pos, neg = costs.split_scores(df)
    return pd.DataFrame([
        {"design": "A. do nothing", **costs.design_cost(pos, neg, NEVER, NEVER)},
        {"design": "B. a person checks every snapshot", **costs.design_cost(pos, neg, 0.0, NEVER)},
        {"design": f"C. model alone (cut {CUT:.2f})", **costs.design_cost(pos, neg, CUT, CUT)},
        {"design": f"D. model + review [{LOW:.2f}, {HIGH:.2f})", **costs.design_cost(pos, neg, LOW, HIGH)},
    ]).set_index("design")


designs_val, designs_test = four_designs(val_df), four_designs(test_df)
COLS = ["eur_per_1000", "eur_per_day", "fn_per_1000", "fp_per_1000", "recall_system", "review_share",
        "reviews_per_day", "review_hours_per_day"]
print(f"unconstrained cheapest band: [{cheapest['low']}, {cheapest['high']}) -> {cheapest['eur_per_1000']:.2f}")
print(f"chosen (fits {costs.REVIEW_HOURS_AVAILABLE} h/day): [{LOW}, {HIGH})\n")
print("VALIDATION\n", designs_val[COLS].round(3).to_string())
print("\nTEST (thresholds from validation)\n", designs_test[COLS].round(3).to_string())

rows = []
for c_rev in [0.05, 0.15, 0.30, 0.50, 1.00]:
    r = best_band(val_df, c_review=c_rev)[1]
    rows.append({"EUR per review": c_rev, "low": r["low"], "high": r["high"], "review share": r["review_share"], "EUR/1,000": r["eur_per_1000"]})
for hr in [0.98, 0.80, 0.70, 0.60]:
    r = best_band(val_df, hr=hr)[1]
    rows.append({"human recall": hr, "low": r["low"], "high": r["high"], "review share": r["review_share"], "EUR/1,000": r["eur_per_1000"]})
sens_review = pd.DataFrame(rows)
print("\n", sens_review.round(3).to_string(index=False))

### Production-like check

Half of the positive test images are scenes where **nobody** wears a helmet (crowds, streets), which are easy and not what a site
camera sees. Re-running designs C and D keeping only the positives that look like a site (someone without a helmet among
colleagues with one), and only one such head, says how much the headline cost depends on those easy images.

In [ ]:
cm_img = costs.confusion(test_df["positive"], test_df["score"], CUT)
scene_recall = (test_df[test_df["positive"] == 1].assign(caught=lambda d: d["score"] >= CUT)
                .groupby("scene")["caught"].agg(["mean", "count"]))

rows = []
for name, df in [("valid", val_df), ("test", test_df)]:
    subsets = {
        "all positives": df,
        "positives among helmets": df[(df["positive"] == 0) | (df["n_helmet"] > 0)],
        "one head among helmets": df[(df["positive"] == 0) | ((df["n_helmet"] > 0) & (df["n_head"] == 1))],
    }
    for label, sub in subsets.items():
        pos, neg = costs.split_scores(sub)
        for design, (lo, hi) in {"C": (CUT, CUT), "D": (LOW, HIGH)}.items():
            r = costs.design_cost(pos, neg, lo, hi)
            rows.append({"split": name, "positives": label, "n positives": len(pos), "design": design,
                         "recall_system": r["recall_system"], "eur_per_1000": r["eur_per_1000"]})
production_like = pd.DataFrame(rows)

print(scene_recall.round(3).to_string(), "\n")
print(production_like.round(3).to_string(index=False))

### The images behind the errors

Worst misses (positives with the lowest score) and the most confident false alarms in test. Red = true `head`, green = true
`helmet`, blue = predicted `head` at conf ≥ 0.25. A few are copied to `examples/` for the dashboard.

In [ ]:
def draw(image, rows, color, width=3):
    W, H = image.size
    d = ImageDraw.Draw(image)
    for xc, yc, w, h in rows:
        d.rectangle([(xc - w / 2) * W, (yc - h / 2) * H, (xc + w / 2) * W, (yc + h / 2) * H], outline=color, width=width)
    return image


def annotated(i):
    row = test_df.iloc[i]
    im = Image.open(row["path"]).convert("RGB")
    b = boxes[(boxes["split"] == "test") & (boxes["image"] == row["image"])]
    draw(im, b.loc[b["cls"] == "helmet", ["xc", "yc", "w", "h"]].to_numpy(), "lime")
    draw(im, b.loc[b["cls"] == "head", ["xc", "yc", "w", "h"]].to_numpy(), "red")
    pb = test_pred_boxes[i]
    keep = (pb["cls"] == HEAD) & (pb["conf"] >= DET_CONF)
    return draw(im, pb["xywhn"][keep], "blue", 2)


def sheet(items, thumb=300, columns=4):
    rows = (len(items) + columns - 1) // columns
    out = Image.new("RGB", (columns * thumb, rows * (thumb + 22)), "white")
    d = ImageDraw.Draw(out)
    font = ImageFont.load_default()
    for k, (caption, im) in enumerate(items):
        x, y = (k % columns) * thumb, (k // columns) * (thumb + 22)
        d.text((x + 4, y + 4), caption, fill="black", font=font)
        out.paste(im.resize((thumb, thumb)), (x, y + 22))
    return out


worst_fn = test_df[test_df["positive"] == 1].nsmallest(8, "score").index
worst_fp = test_df[test_df["positive"] == 0].nlargest(8, "score").index
sheet([(f"missed, score {test_df.loc[i, 'score']:.2f}", annotated(i)) for i in worst_fn]).save(ART_DIR / "failures_missed.png")
sheet([(f"false alarm, score {test_df.loc[i, 'score']:.2f}", annotated(i)) for i in worst_fp]).save(ART_DIR / "failures_false_alarms.png")

# examples for the upload box: two clear cases each way, the worst miss and the worst false alarm
for f in EXAMPLES_DIR.glob("*.jpg"):
    f.unlink()
def pick(mask, largest):
    sub = test_df[mask]
    return None if sub.empty else (sub["score"].idxmax() if largest else sub["score"].idxmin())


pos, neg = test_df["positive"] == 1, test_df["positive"] == 0
picks = {
    "positive_among_helmets": pick(pos & (test_df["n_helmet"] > 0), True),
    "positive_nobody_helmet": pick(pos & (test_df["n_helmet"] == 0), True),
    "negative_clear": pick(neg, False),
    "negative_crew": pick(neg & (test_df["n_helmet"] >= 4), False),
    "missed_violation": pick(pos, False),
    "false_alarm": pick(neg, True),
}
for name, i in picks.items():
    if i is None:
        continue
    Image.open(test_df.loc[i, "path"]).convert("RGB").save(EXAMPLES_DIR / f"{name}.jpg", quality=90)
print(sorted(p.name for p in EXAMPLES_DIR.iterdir()))

## 9. Save everything the dashboard needs

The dashboard never trains: it loads `weights/hardhat_yolo11n.pt` for uploaded images, and recomputes the confusion matrix and the
costs from the per-snapshot scores when the sliders move.

In [ ]:
keep = ["image", "positive", "n_head", "n_helmet", "scene", "score", "coco_score"]
val_df[keep].to_csv(ART_DIR / "scores_valid.csv", index=False)
test_df[keep].to_csv(ART_DIR / "scores_test.csv", index=False)
per_class.to_csv(ART_DIR / "per_class_test.csv")
cm_df.to_csv(ART_DIR / "confusion_matrix_test.csv")
inst.to_csv(ART_DIR / "dataset_instances.csv")
img_level.to_csv(ART_DIR / "dataset_image_level.csv")
scenes.to_csv(ART_DIR / "dataset_scenes.csv")
heads_bucket.to_csv(ART_DIR / "dataset_heads_per_image.csv")
size.to_csv(ART_DIR / "dataset_object_size.csv")
baselines.to_csv(ART_DIR / "baselines_image_level.csv", index=False)
det_table.to_csv(ART_DIR / "baselines_detection.csv")
pd.concat([sweep_val.assign(split="valid"), sweep_test.assign(split="test")]).to_csv(ART_DIR / "cost_sweep.csv", index=False)
pd.concat([designs_val.assign(split="valid"), designs_test.assign(split="test")]).to_csv(ART_DIR / "designs.csv")
gt_heads.to_csv(ART_DIR / "head_recall_by_size_test.csv", index=False)
base_rate_table.to_csv(ART_DIR / "sensitivity_base_rate_precision.csv", index=False)
sens_fn.to_csv(ART_DIR / "sensitivity_fn_cost.csv", index=False)
sens_p.to_csv(ART_DIR / "sensitivity_base_rate.csv", index=False)
sens_review.to_csv(ART_DIR / "sensitivity_review.csv", index=False)
production_like.to_csv(ART_DIR / "production_like.csv", index=False)
scene_recall.to_csv(ART_DIR / "image_recall_by_scene_test.csv")

for f, folder in [("confusion_matrix.png", CM_DIR), ("confusion_matrix_normalized.png", CM_DIR), ("BoxPR_curve.png", TEST_DIR)]:
    if (folder / f).exists():
        shutil.copy(folder / f, ART_DIR / f)

config = {
    "dataset": {"workspace": WORKSPACE, "project": PROJECT, "version": VERSION,
                "url": f"https://universe.roboflow.com/{WORKSPACE}/{PROJECT}/dataset/{VERSION}"},
    "classes": CLASS_NAMES, "positive_class": "head",
    "training": TRAIN_SETTINGS,
    "business": costs.params(),
    "operating_point": {"single_cut": CUT, "low": LOW, "high": HIGH, "chosen_on": "valid",
                        "f1_cut": F1_CUT, "accuracy_cut": ACC_CUT},
    "test_metrics": overall,
    "image_confusion_test_at_cut": cm_img,
}
(ART_DIR / "config.json").write_text(json.dumps(config, indent=2, default=float))
print(sorted(p.name for p in ART_DIR.iterdir()))